# Data Engineering Interview Playground
## SQL + Python with PostgreSQL and Jupyter

This notebook provides an interactive environment to practice SQL queries, Python data transformations, and ETL pipelines with PostgreSQL backend.

**Features:**
- Execute SQL queries and work with results as pandas DataFrames
- Run Python code with access to SQL query results
- Test solutions against gold-standard expected outputs
- Practice real-world data engineering interview questions

## Section 1: Environment Setup and Dependencies

Install and import all required libraries for SQL execution, data processing, and database connections.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install pandas psycopg2-binary sqlalchemy ipython-sql python-dotenv

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
import json
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

print("✅ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Section 2: Database Connection Configuration

Configure PostgreSQL connection using SQLAlchemy. Update credentials based on your PostgreSQL instance.

In [ ]:
# Database Configuration
DB_USER = os.getenv("DB_USER", "username")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "interview_db")

# Create connection string
DB_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Create SQLAlchemy engine
try:
    engine = create_engine(DB_URL)
    # Test connection
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(f"✅ Successfully connected to PostgreSQL")
    print(f"   Database: {DB_NAME} @ {DB_HOST}:{DB_PORT}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print(f"   Make sure PostgreSQL is running at {DB_HOST}:{DB_PORT}")

## Section 3: SQL Query Execution and Result Handling

Create functions to execute SQL queries, handle results as DataFrames, and display with error handling.

In [ ]:
def execute_sql(query: str, display=True) -> pd.DataFrame:
    """Execute a SQL query and return results as a pandas DataFrame."""
    try:
        with engine.connect() as conn:
            df = pd.read_sql(text(query), conn)
        if display:
            print(f"✅ Query executed successfully. Rows returned: {len(df)}")
        return df
    except Exception as e:
        print(f"❌ Query execution failed: {e}")
        return pd.DataFrame()

def show_query_info(df: pd.DataFrame):
    """Display information about a query result DataFrame."""
    print(f"\nDataFrame Info:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Data types:\n{df.dtypes}")
    print(f"\nFirst few rows:")
    return df

# Example: Execute a basic query
print("Example SQL Query:")
example_query = "SELECT * FROM users LIMIT 3;"
print(f"Query: {example_query}")
result_df = execute_sql(example_query, display=True)

## Section 4: Python Code Execution with DataFrame Integration

Implement safe code execution with controlled namespaces to reference SQL results.

In [ ]:
def execute_python_with_dataframe(code: str, input_df: pd.DataFrame = None):
    """
    Execute Python code with access to input DataFrame.
    Use 'df' variable to reference the input DataFrame.
    Use 'result' variable to store output.
    """
    try:
        # Set up namespace with required libraries
        namespace = {
            'pd': pd,
            'np': np,
            'datetime': datetime,
            'timedelta': timedelta,
            'json': json,
            'df': input_df if input_df is not None else pd.DataFrame()
        }
        
        # Execute the code
        exec(code, namespace)
        
        # Extract result
        result = namespace.get('result', None)
        print("✅ Python code executed successfully!")
        return result
    except Exception as e:
        print(f"❌ Execution error: {e}")
        return None

# Example: Transform SQL results with Python
if not result_df.empty:
    python_code = """
# Transform the SQL result
result = result_df.groupby('region').size().reset_index(name='count')
result = result.sort_values('count', ascending=False)
"""
    
    print("Example Python Transformation:")
    transformed = execute_python_with_dataframe(python_code, result_df)
    if transformed is not None:
        print(transformed)

## Section 5: Accuracy Testing Framework

Build a testing function that compares user results against gold-standard expected outputs.

In [ ]:
def test_accuracy(actual_result: pd.DataFrame, expected_result: pd.DataFrame, test_name: str = "Test") -> bool:
    """
    Compare actual results against expected results.
    Returns True if they match, False otherwise.
    """
    
    print(f"\n{'='*60}")
    print(f"Running Accuracy Test: {test_name}")
    print(f"{'='*60}")
    
    # Check if both are DataFrames
    if not isinstance(actual_result, pd.DataFrame) or not isinstance(expected_result, pd.DataFrame):
        print(f"❌ FAIL - Results must be pandas DataFrames")
        return False
    
    # Check shape
    if actual_result.shape != expected_result.shape:
        print(f"❌ FAIL - Shape mismatch")
        print(f"   Expected shape: {expected_result.shape}")
        print(f"   Actual shape: {actual_result.shape}")
        return False
    
    # Check columns
    if set(actual_result.columns) != set(expected_result.columns):
        print(f"❌ FAIL - Column mismatch")
        print(f"   Expected columns: {list(expected_result.columns)}")
        print(f"   Actual columns: {list(actual_result.columns)}")
        return False
    
    # Sort both dataframes by all columns for comparison
    try:
        actual_sorted = actual_result.sort_values(by=list(actual_result.columns)).reset_index(drop=True)
        expected_sorted = expected_result.sort_values(by=list(expected_result.columns)).reset_index(drop=True)
        
        if actual_sorted.equals(expected_sorted):
            print(f"✅ PASS - Results match expected output!")
            print(f"   Rows: {len(actual_result)}")
            return True
        else:
            print(f"❌ FAIL - Data mismatch")
            print(f"\nExpected:")
            print(expected_result.head(10))
            print(f"\nActual:")
            print(actual_result.head(10))
            return False
    except Exception as e:
        print(f"❌ FAIL - Comparison error: {e}")
        return False

# Example test
print("Example: Testing a simple transformation")
expected_data = pd.DataFrame({'region': ['North America', 'Europe', 'Asia'], 'count': [3, 2, 2]})
actual_data = result_df.groupby('region').size().reset_index(name='count').sort_values('region').reset_index(drop=True)
test_accuracy(actual_data, expected_data, "Regional Count Test")

## Section 6: Sample Interview Questions

Practice real-world data engineering interview problems with starter code and validation.

### Question 1: Window Functions - Rank Top Spenders by Month

**Problem:** Find the top 3 spenders in each month using window functions.

**Starter SQL:**
```sql
SELECT 
    EXTRACT(YEAR_MONTH FROM transaction_date) as month,
    user_id,
    SUM(amount) as total_spent,
    RANK() OVER (PARTITION BY EXTRACT(YEAR_MONTH FROM transaction_date) ORDER BY SUM(amount) DESC) as rank
FROM transactions
GROUP BY month, user_id
HAVING RANK() OVER (...) <= 3
ORDER BY month, rank;
```

In [ ]:
# Q1: Your Solution
q1_solution = """
SELECT 
    DATE_TRUNC('month', transaction_date)::DATE as month,
    user_id,
    SUM(amount) as total_spent,
    RANK() OVER (PARTITION BY DATE_TRUNC('month', transaction_date) ORDER BY SUM(amount) DESC) as rank
FROM transactions
GROUP BY DATE_TRUNC('month', transaction_date), user_id
ORDER BY month DESC, rank ASC
LIMIT 20;
"""

print("Q1: Top Spenders by Month")
q1_result = execute_sql(q1_solution)
if not q1_result.empty:
    print(q1_result)

### Question 2: Deduplication - Keep Latest User Record

**Problem:** The `user_history` table has duplicate user records with different timestamps. Keep only the most recent record per user.

**Expected Output:**
- One record per user_id
- The record with the most recent `updated_at` timestamp

In [ ]:
# Q2: Your Solution
q2_solution = """
SELECT DISTINCT ON (user_id) 
    user_id, 
    email, 
    updated_at
FROM user_history
ORDER BY user_id, updated_at DESC;
"""

print("Q2: Deduplicate User History (Keep Latest)")
q2_result = execute_sql(q2_solution)
if not q2_result.empty:
    print(q2_result)

### Question 3: SQL Analytical - Total Revenue by Region

**Problem:** Calculate total revenue by region from the transactions table.

**Expected Output:**
- One row per region
- Columns: region, total_revenue
- Sorted by total_revenue descending

In [ ]:
# Q3: Your Solution
q3_solution = """
SELECT 
    region,
    SUM(amount) as total_revenue,
    COUNT(*) as num_transactions,
    AVG(amount) as avg_transaction
FROM transactions
GROUP BY region
ORDER BY total_revenue DESC;
"""

print("Q3: Total Revenue by Region")
q3_result = execute_sql(q3_solution)
if not q3_result.empty:
    print(q3_result)

## Section 7: ETL Pipeline Example

Create a complete ETL pipeline: Extract from SQL, Transform with Python, Load as JSON.

### ETL Task: Tax Calculation Pipeline

**Scenario:** Calculate sales tax for recent transactions and generate a summary report.

**Requirements:**
1. Extract all transactions from the last 30 days
2. Calculate tax based on region (different tax rates)
3. Generate summary statistics
4. Output as JSON

In [ ]:
# EXTRACT: Get recent transactions from the last 30 days
extract_query = """
SELECT 
    transaction_id,
    user_id,
    amount,
    transaction_date,
    region
FROM transactions
WHERE transaction_date >= CURRENT_DATE - INTERVAL '30 days'
ORDER BY transaction_date DESC;
"""

print("=" * 60)
print("ETL PIPELINE: TAX CALCULATION")
print("=" * 60)
print("\nSTEP 1: EXTRACT")
transactions_df = execute_sql(extract_query)
print(f"Extracted {len(transactions_df)} transactions")

# TRANSFORM: Calculate tax based on region
print("\nSTEP 2: TRANSFORM")

transform_code = """
# Define tax rates by region
tax_rates = {
    'North America': 0.08,
    'Europe': 0.20,
    'Asia': 0.10,
    'South America': 0.15,
    'Africa': 0.12,
    'Oceania': 0.10
}

# Add tax calculations
df['tax_rate'] = df['region'].map(tax_rates)
df['tax_amount'] = (df['amount'] * df['tax_rate']).round(2)
df['total_with_tax'] = (df['amount'] + df['tax_amount']).round(2)

# Create summary statistics
result = {
    'total_revenue': df['amount'].sum(),
    'total_tax_collected': df['tax_amount'].sum(),
    'total_with_tax': df['total_with_tax'].sum(),
    'avg_transaction': df['amount'].mean(),
    'num_transactions': len(df),
    'by_region': df.groupby('region').agg({
        'amount': 'sum',
        'tax_amount': 'sum',
        'total_with_tax': 'sum'
    }).round(2).to_dict()
}
"""

etl_result = execute_python_with_dataframe(transform_code, transactions_df)
print(f"Transformed {len(transactions_df)} transactions with tax calculations")

# LOAD: Output as JSON
print("\nSTEP 3: LOAD")
if etl_result:
    print("JSON Output:")
    print(json.dumps(etl_result, indent=2, default=str))
    
    # Save to file
    output_file = "etl_output.json"
    with open(output_file, 'w') as f:
        json.dump(etl_result, f, indent=2, default=str)
    print(f"\n✅ Pipeline complete! Output saved to {output_file}")

## Next Steps and Advanced Topics

### 📚 Practice Areas
- **Window Functions**: RANK(), ROW_NUMBER(), LAG(), LEAD(), PARTITION BY
- **Joins**: INNER, LEFT, RIGHT, FULL OUTER, CROSS
- **Aggregations**: GROUP BY, HAVING, Complex nested queries
- **CTEs**: WITH clauses and recursive queries
- **Data Cleaning**: NULL handling, deduplication, outlier detection
- **Performance**: Query optimization, indexing strategies

### 🚀 Additional Exercises
1. **Cohort Analysis** - Analyze user behavior over time
2. **RFM Segmentation** - Recency, Frequency, Monetary analysis
3. **Time Series Analysis** - Trends and forecasting
4. **Data Validation** - Quality checks and data profiling
5. **Complex ETL** - Multi-step transformations with validation

### 💡 Tips for Interview Success
- Write clean, readable queries with proper formatting
- Always test your SQL with sample data
- Consider edge cases and NULL values
- Use CTEs to break down complex queries
- Validate your results before submitting
- Explain your approach and reasoning

### 📖 Resources
- PostgreSQL Documentation: https://www.postgresql.org/docs/
- SQL Window Functions: https://www.postgresql.org/docs/current/functions-window.html
- Pandas Documentation: https://pandas.pydata.org/docs/
- SQLAlchemy: https://docs.sqlalchemy.org/